to add custom penalty to source code instead of wrapper:

```python
if reward == 0:
    reward = 0
```

replace with 

```python
if reward == 0:
    reward -= 0.01
else:
    fy, fx = self.food.getFoodCoordinates()
    distance = abs(fy - y) + abs(fx - x)
    reward -= 0.001 * distance  # bigger penalty the further away it is
```

you can use wrapper, its more flexible you can add params to sweep

In [ ]:
from typing import Any, SupportsFloat
import pygame
import random
import numpy as np
import time
import keyboard
import copy
import numpy
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecEnv
from gymnasium.wrappers import TimeLimit
import cv2
from gymnasium.envs.registration import register


class SnakeEnv:

    def __init__(self, height, width) -> None:
        self.height = height
        self.width = width
        self.observationSpace = [
            [0 for _ in range(self.width)] for _ in range(self.height)]
        pass

    def getValue(self, y, x):
        return self.observationSpace[y][x]

    def setValue(self, y, x, value):
        if self.observationSpace[y][x] != value:
            self.observationSpace[y][x] = value

    def setNull(self, y, x):
        self.observationSpace[y][x] = 0

    def print(self):
        print()
        for y in range(len(self.observationSpace)):
            print(self.observationSpace[y])

    def getDesignationValue(self, value):
        result = []
        for y in range(len(self.observationSpace)):
            for x in range(len(self.observationSpace[y])):
                if self.observationSpace[y][x] == value:
                    result.append((y, x))
        return result

    def getEnvironmentalData(self):
        return self.observationSpace


import random
from collections import deque

class Wall:
    def __init__(self, snakeEnv: SnakeEnv, num_interior_walls: int = 2) -> None:
        self.value = 1
        self.snakeEnv = snakeEnv
        self.walls = []
        self.num_interior_walls = num_interior_walls
        self.generate()

    def generate(self):
        snakeEnv = self.snakeEnv
        # Always place border walls
        for y in range(snakeEnv.height):
            for x in range(snakeEnv.width):
                if x == 0 or x == snakeEnv.width - 1 or \
                        y == 0 or y == snakeEnv.height - 1:
                    snakeEnv.setValue(y, x, self.value)
                    self.walls.append((y, x))

        # Try to place interior walls randomly
        placed = 0
        attempts = 0
        max_attempts = self.num_interior_walls * 20  # avoid infinite loops

        while placed < self.num_interior_walls and attempts < max_attempts:
            attempts += 1
            # Pick a random interior cell
            y = random.randint(1, snakeEnv.height - 2)
            x = random.randint(1, snakeEnv.width - 2)

            if self.isWall(y, x):
                continue

            # Tentatively add wall and check safety
            self.walls.append((y, x))
            if self._is_safe():
                snakeEnv.setValue(y, x, self.value)
                placed += 1
            else:
                # Reject — would create a deadlock cell
                self.walls.pop()

    def _is_safe(self) -> bool:
        """
        Check two conditions:
        1. All non-wall cells are still connected (no isolated regions).
        2. No non-wall cell has fewer than 2 non-wall neighbours
           (a cell with only 1 free neighbour is a dead-end that
           prevents a Hamiltonian path on the interior grid).
        """
        snakeEnv = self.snakeEnv
        wall_set = set(self.walls)

        free_cells = [
            (y, x)
            for y in range(snakeEnv.height)
            for x in range(snakeEnv.width)
            if (y, x) not in wall_set
        ]

        if not free_cells:
            return False

        # connectivity via BFS
        visited = set()
        queue = deque([free_cells[0]])
        visited.add(free_cells[0])

        while queue:
            cy, cx = queue.popleft()
            for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]:
                nb = (cy + dy, cx + dx)
                if nb not in visited and nb not in wall_set \
                        and 0 <= nb[0] < snakeEnv.height \
                        and 0 <= nb[1] < snakeEnv.width:
                    visited.add(nb)
                    queue.append(nb)

        if len(visited) != len(free_cells):
            return False  # disconnected region

        #no dead-end cells
        for (cy, cx) in free_cells:
            free_neighbours = sum(
                1 for dy, dx in [(-1,0),(1,0),(0,-1),(0,1)]
                if (cy + dy, cx + dx) not in wall_set
                and 0 <= cy + dy < snakeEnv.height
                and 0 <= cx + dx < snakeEnv.width
            )
            if free_neighbours < 2:
                return False  # dead-end cell -> Hamiltonian path impossible

        return True

    def isWall(self, y, x):
        return (y, x) in set(self.walls)



    def reset(self):
        snakeEnv = self.snakeEnv

        # Clear only interior walls from the grid
        for (y, x) in self.walls:
            if not self._is_border(y, x):
                snakeEnv.setValue(y, x, 0)

        # Keep only border walls
        self.walls = [(y, x) for (y, x) in self.walls if self._is_border(y, x)]

        # Re-place border walls on the fresh grid
        for (y, x) in self.walls:
            snakeEnv.setValue(y, x, self.value)

        # Re-run interior placement
        placed = 0
        attempts = 0
        max_attempts = self.num_interior_walls * 20

        while placed < self.num_interior_walls and attempts < max_attempts:
            attempts += 1
            y = random.randint(1, snakeEnv.height - 2)
            x = random.randint(1, snakeEnv.width - 2)

            if self.isWall(y, x):
                continue

            self.walls.append((y, x))
            if self._is_safe():
                snakeEnv.setValue(y, x, self.value)
                placed += 1
            else:
                self.walls.pop()
        
    def _is_border(self, y, x) -> bool:
        snakeEnv = self.snakeEnv
        return x == 0 or x == snakeEnv.width - 1 or \
            y == 0 or y == snakeEnv.height - 1


class Food:

    def __init__(self, snakeEnv: SnakeEnv, seed=None) -> None:
        self.value = 2
        self.snakeEnv = snakeEnv
        if seed == None:
            seed = int(time.time())
        random.seed(seed)
        self.foodCoordinates = ()
        self.generate()
        pass

    def generate(self, remove=False):
        if remove and len(self.foodCoordinates) == 2:
            y, x = self.foodCoordinates
            self.snakeEnv.setNull(y, x)
        emptyPosition = self.snakeEnv.getDesignationValue(0)
        lenEmptyPosition = len(emptyPosition)
        if lenEmptyPosition <= 0:
            return True
        if lenEmptyPosition == 1:
            y, x = emptyPosition[0]
        else:
            ri = random.randint(0, len(emptyPosition) - 1)
            y, x = emptyPosition[ri]
        self.foodCoordinates = (y, x)
        self.snakeEnv.setValue(y, x, self.value)
        return False

    def getFoodCoordinates(self):
        return self.foodCoordinates

    def isFood(self, y, x):
        vy, vx = self.foodCoordinates
        return vy == y and vx == x

    def rendering(self):
        if len(self.foodCoordinates) == 2:
            y, x = self.foodCoordinates
            self.snakeEnv.setValue(y, x, self.value)


class SnakeBody:

    UP = 1
    DOWN = 2
    LEFT = 3
    RIGHT = 4

    def __init__(self, snakeEnv: SnakeEnv) -> None:
        self.snakeEnv = snakeEnv
        self.head_value = 3
        self.body_value = 4
        # Length
        self.len = 1
        # Direction of movement
        self.movingDirection = 1
        self.body = [(int(snakeEnv.height / 2), int(snakeEnv.width / 2))]
        self.end = ()
        self.generate()
        pass

    def getBody(self):
        return self.body

    def getHead(self):
        return self.body[0]

    def length(self):
        return len(self.body)

    def setAction(self, action):
        if action == 0:
            return
        if (action % 2 == 0 and action - 1 == self.movingDirection) or \
                (action % 1 == 0 and action + 1 == self.movingDirection):
            return
        self.movingDirection = action

    def lengthen(self):
        if len(self.end) == 2:
            y, x = self.end
            self.body.append((y, x))
            self.snakeEnv.setValue(y, x, self.body_value)
            self.end = ()

    def move(self):
        y, x = self.body[0]
        if self.movingDirection == SnakeBody.UP:
            y -= 1
        elif self.movingDirection == SnakeBody.DOWN:
            y += 1
        elif self.movingDirection == SnakeBody.LEFT:
            x -= 1
        elif self.movingDirection == SnakeBody.RIGHT:
            x += 1
        else:
            raise RuntimeError(f"Exception move : {self.movingDirection}")
        self.body.insert(0, (y, x))
        ey, ex = self.body.pop()
        self.end = (ey, ex)
        self.snakeEnv.setNull(ey, ex)
        self.generate()
        return (y, x)

    def headBodyStack(self):
        hy, hx = self.body[0]
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            if hy == y and hx == x:
                return True
        return False

    def isBody(self, hy, hx):
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            if hy == y and hx == x:
                return True
        return False

    def generate(self):
        y, x = self.body[0]
        self.snakeEnv.setValue(y, x, self.head_value)
        for i in range(1, len(self.body)):
            y, x = self.body[i]
            self.snakeEnv.setValue(y, x, self.body_value)


class TcsV2Env(gym.Env):

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, **kwargs) -> None:
        self.image_size = kwargs.get("image_size", 10)
        self.show = False
        self.height = kwargs.get("height", 50)
        self.width = kwargs.get("width", 50)
        self.windowcolor = [(0, 0, 0), (255, 255, 255), (255, 0, 0),
                            (0, 0, 255), None]
        self.WINDOW_WIDTH = self.width * self.image_size
        self.WINDOW_HEIGHT = self.height * self.image_size
        self.WINDOW_BLACK = (0, 0, 0)
        self.obs_type = kwargs.get("obs_type", "rgb")
        self.outputType = kwargs.get("outputType", "Discrete")
        if self.outputType == "Discrete":
            self.action_space = gym.spaces.Discrete(5)
        elif self.outputType == "Box":
            self.action_space = gym.spaces.Box(
                low=0, high=4, shape=(1,), dtype=np.uint8
            )
        if self.obs_type == "rgb":
            self.observation_space = gym.spaces.Box(
                low=0, high=255, shape=(self.height * self.image_size, self.width * self.image_size, 3), dtype=np.uint8)
        if self.obs_type == "data":
            self.observation_space = gym.spaces.Box(
                low=0, high=255, shape=(8,), dtype=np.float32)
        self.render_mode = kwargs.get("render_mode", None)
        self.mp4 = kwargs.get("mp4", False)
        self.mp4fps = kwargs.get("mp4fps", 30)
        assert self.render_mode is None or self.render_mode in self.metadata["render_modes"]
        super().__init__()

    def get_action_meanings(self):
        return ["NOOP", "UP", "DOWN", "LEFT", "RIGHT"]

    def step(self, action: Any) -> tuple[Any, SupportsFloat, bool, bool, dict[str, Any]]:
        if isinstance(action, np.ndarray):
            action = action[0]
            action = round(action)
        info = {}
        reward = 0
        self.snakebody.setAction(action)
        y, x = self.snakebody.move()
        done = False
        if self.snakebody.isBody(y, x):
            done = True
            info["message"] = "Come into contact with the body"
        if self.wall.isWall(y, x):
            done = True
            info["message"] = "hit the wall"
        if self.food.isFood(y, x):
            self.snakebody.lengthen()
            if self.food.generate():
                done = True
                vn_step = 1000 - self.num_step
                if vn_step < 0:
                    vn_step = 0
                reward += 100 * (vn_step / 1000)
                info["message"] = "Damn, the screen is completely covered with it"
                if self.show:
                    time.sleep(3)
            reward += 1
            # self.distanceFood = 20
        # else:
        #     fy, fx = self.food.getFoodCoordinates()
        #     jy = abs(fy - y)
        #     jx = abs(fx - x)
        #     distance = jy + jx
        #     if distance < self.distanceFood:
        #         self.distanceFood = distance
        #         reward += 1 - ((1 / 20) * distance)
        if reward < 0:
            reward = 0
       
            
        self.score += reward
        observation = self.environmentalData()

        if self.mp4 and self.obs_type != "rgb":
            self.render()

        self.num_step += 1
        return observation, reward, done, False, info

    def reset(self, *, seed: int | None = None, options: dict[str, Any] | None = None) -> tuple[Any, dict[str, Any]]:
        self.snakeEnv = SnakeEnv(self.height, self.width)
       
        if hasattr(self, 'wall'):
            self.wall.snakeEnv = self.snakeEnv  # point wall to the fresh grid
            self.wall.reset()
        else:
            self.wall = Wall(self.snakeEnv)

        self.snakebody = SnakeBody(self.snakeEnv)
        self.food = Food(self.snakeEnv, seed=seed)
        observation = self.environmentalData()
        self.score = 0
        self.num_step = 0
        # self.distanceFood = 20
        return observation, {}


    # Changed observations, added danger observations  
    # new observation space 225 * 16 * 4 * 36 = 518_400 rows
    def environmentalData(self):
        
        if self.obs_type == "data":
            hy, hx = self.snakebody.getHead()
            fy, fx = self.food.getFoodCoordinates()

            # Relative food position (more signal)
            food_dy = fy - hy
            food_dx = fx - hx

            # Danger flags in each direction
            danger_up = int(self.wall.isWall(hy-1, hx) or self.snakebody.isBody(hy-1, hx))
            danger_down = int(self.wall.isWall(hy+1, hx) or self.snakebody.isBody(hy+1, hx))
            danger_left = int(self.wall.isWall(hy, hx-1) or self.snakebody.isBody(hy, hx-1))
            danger_right = int(self.wall.isWall(hy, hx+1) or self.snakebody.isBody(hy, hx+1))

            # Current direction (1–4)
            direction = self.snakebody.movingDirection

            # Snake length (normalised)
            length = self.snakebody.length()

            return np.array([food_dy, food_dx,
                 danger_up, danger_down, danger_left, danger_right,
                 direction,
                 length], dtype=np.float32)
        
        elif self.obs_type == "rgb":
            observation = self.render()
            return observation
        
        return
    def render(self):
        if (self.render_mode == "human") and ("window" not in self.__dict__ or self.window == None):
            pygame.init()
            self.window = pygame.display.set_mode(
                (self.width * 10, self.height * 10))
            pygame.display.set_caption("tcs")
            self.window.fill(self.WINDOW_BLACK)
            self.show = True
        # Create a canvas
        if "canvas" not in self.__dict__ or self.canvas == None:
            self.canvas = pygame.Surface(
                (self.WINDOW_WIDTH, self.WINDOW_HEIGHT))
        if self.show and ("canvasN" not in self.__dict__ or self.canvasN == None):
            self.canvasN = pygame.Surface(
                (self.width * 10, self.height * 10)
            )
        self.canvas.fill(self.WINDOW_BLACK)
        if self.show:
            self.canvasN.fill(self.WINDOW_BLACK)

        observation = self.snakeEnv.getEnvironmentalData()
        for y in range(len(observation)):
            for x in range(len(observation[y])):
                value = observation[y][x]
                if value == 0 or self.windowcolor[value] == None:
                    continue
                pygame.draw.rect(
                    self.canvas, self.windowcolor[value], (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
                if self.show:
                    pygame.draw.rect(
                        self.canvasN, self.windowcolor[value], (
                            x * 10, y * 10, 10, 10)
                    )
        body = self.snakebody.getBody()
        bodylen = len(body)
        if bodylen > 1:
            colosNm = round(105 / (bodylen - 1))
            dcolosNm = 150 + colosNm
            for i in range(bodylen - 2, 0, -1):
                y, x = body[i]
                vvColos = (0, dcolosNm, 0)
                pygame.draw.rect(
                    self.canvas, vvColos, (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
                if self.show:
                    pygame.draw.rect(
                        self.canvasN, vvColos, (
                            x * 10, y * 10, 10, 10)
                    )
                if dcolosNm >= 255:
                    continue
                dcolosNm += colosNm
                if dcolosNm > 255:
                    dcolosNm = 255
            y, x = body[-1]
            vbcolos = (232, 255, 1)
            pygame.draw.rect(
                self.canvas, vbcolos, (x * self.image_size, y * self.image_size, self.image_size, self.image_size))
            if self.show:
                pygame.draw.rect(
                    self.canvasN, vbcolos, (
                        x * 10, y * 10, 10, 10)
                )

        result = np.transpose(
            np.array(pygame.surfarray.pixels3d(self.canvas)), axes=(1, 0, 2)
        )
        if self.render_mode == "human":
            show_image = np.transpose(
                np.array(pygame.surfarray.pixels3d(self.canvasN)), axes=(1, 0, 2)
            )
            self.window.blit(self.canvasN, self.canvasN.get_rect())
            # Cleanup Event
            pygame.event.pump()
            pygame.display.update()
            # pygame.display.flip()
            # event = pygame.event.poll()
            # # pygame.display.update()
            # if event.type == pygame.QUIT:
            #     pygame.quit()
            #     exit()
            if self.mp4:
                if "outMp4" not in self.__dict__ or self.outMp4 == None:
                    height, width, _ = show_image.shape
                    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                    self.outMp4 = cv2.VideoWriter(
                        "1.mp4", fourcc, self.mp4fps, (width, height))
                self.outMp4.write(show_image)

        return result

    def close(self):
        del self.snakebody
        del self.wall
        del self.food
        del self.snakeEnv
        del self.score
        if self.mp4 and hasattr(self, "outMp4") and self.outMp4 is not None:
            self.outMp4.release()
        return super().close()


class ConstructEnv:

    def __init__(self) -> None:
        gym.register(id="tcs-v2", entry_point=TcsV2Env, max_episode_steps=2000,
                     kwargs={})
        pass

    def getDummyVecEnvs(self, num=32, **kwargs) -> DummyVecEnv:
        return DummyVecEnv([lambda: self.getEnv(**kwargs) for _ in range(num)])

    def getSudoVecEnvs(self, num=6, **kwargs) -> SubprocVecEnv:
        return SubprocVecEnv([lambda: self.getEnv(**kwargs) for _ in range(num)])

    def getEnv(self, **kwargs) -> TimeLimit:
        return TimeLimit(gym.make("tcs-v2", **kwargs), max_episode_steps=5000)




register(
    id="tcs-v2", entry_point=TcsV2Env, max_episode_steps=2000,
                     kwargs={}
)

In [ ]:
import torch 
import torch.nn as nn
from collections import deque
import random
from stable_baselines3.common.base_class import BaseAlgorithm
from stable_baselines3.common.policies import BasePolicy
import wandb
import numpy as np
from tqdm.notebook import trange


# Dueling Q-Network
class DuelingQNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        
        # Shared feature extractor
        self.feature = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.ReLU()
        )
        
        # Value stream
        self.value = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
        # Advantage stream
        self.advantage = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_actions)
        )

    def forward(self, x):
        features = self.feature(x.float())
        value = self.value(features)
        advantage = self.advantage(features)
        
        return value + (advantage - advantage.mean(dim=1, keepdim=True))

# Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, obs, action, reward, next_obs, done):
        self.buffer.append((obs, action, reward, next_obs, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        obs, actions, rewards, next_obs, dones = zip(*batch)
        return (
            torch.tensor(np.array(obs), dtype=torch.float32),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(np.array(next_obs), dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buffer)

class PrioritisedReplayBuffer:
    def __init__(self, capacity, alpha=0.6, beta_start=0.4, beta_frames=100_000):
        self.capacity = capacity
        self.alpha = alpha          
        self.beta_start = beta_start  
        self.beta_frames = beta_frames
        self.frame = 1

        self.buffer = []
        self.priorities = np.zeros((capacity,), dtype=np.float32)
        self.pos = 0

    def push(self, obs, action, reward, next_obs, done):
        # New transitions get max priority so they're sampled at least once
        max_priority = self.priorities.max() if self.buffer else 1.0

        if len(self.buffer) < self.capacity:
            self.buffer.append((obs, action, reward, next_obs, done))
        else:
            self.buffer[self.pos] = (obs, action, reward, next_obs, done)

        self.priorities[self.pos] = max_priority
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size):
        if len(self.buffer) == self.capacity:
            priorities = self.priorities
        else:
            priorities = self.priorities[:len(self.buffer)]

        # Convert priorities to probabilities
        probs = priorities ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[i] for i in indices]

        # Importance sampling weights to correct for bias
        beta = min(1.0, self.beta_start + self.frame * (1.0 - self.beta_start) / self.beta_frames)
        self.frame += 1

        weights = (len(self.buffer) * probs[indices]) ** (-beta)
        weights /= weights.max()
        weights = torch.tensor(weights, dtype=torch.float32)

        obs, actions, rewards, next_obs, dones = zip(*samples)
        return (
            torch.tensor(np.array(obs), dtype=torch.float32),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(np.array(next_obs), dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32),
            indices,
            weights
        )

    def update_priorities(self, indices, td_errors):
        for idx, td_error in zip(indices, td_errors):
            self.priorities[idx] = abs(td_error) + 1e-6  # small epsilon to avoid zero priority

    def __len__(self):
        return len(self.buffer)

# Q-Learning Algorithm
class QLearning(BaseAlgorithm):

    def __init__(
        self,
        env,
        learning_rate = 1e-3,
        gamma = 0.99, # discount factor
        epsilon_start = 1.0, # exploration start
        epsilon_end = 0.05, # exploration end
        epsilon_decay = 0.99,# epsilon multiplied each episode
        buffer_size = 50_000, # replay buffer capacity
        batch_size = 64,
        train_start = 1_000, # steps before training begins
        target_update = 500, # steps between target network syncs
        alpha = 0.6,
        beta_start = 0.4,
        beta_frames = 100_000,
        use_prioritised_buffer = True,
        early_stopping_patience = 25,
        verbose = 1,
        device = "cpu"
    ):
        super().__init__(
            policy = BasePolicy,
            env = env,
            learning_rate = learning_rate,
            verbose = verbose,
            support_multi_env = True,
            device=device
        )
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.buffer_size = buffer_size
        self.batch_size = batch_size
        self.train_start = train_start
        self.target_update = target_update
        self.alpha = alpha
        self.beta_start = beta_start
        self.beta_frames = beta_frames
        self.use_prioritised_buffer = use_prioritised_buffer
        self.early_stopping_patience = early_stopping_patience

        self._setup_model()

    def _setup_model(self):
        obs_dim   = self.observation_space.shape[0]
        n_actions = self.action_space.n

        self.q_net = DuelingQNetwork(obs_dim, n_actions).to(self.device)
        self.target_q_net = DuelingQNetwork(obs_dim, n_actions).to(self.device)
        self.target_q_net.load_state_dict(self.q_net.state_dict())
        self.target_q_net.eval()

        self.optimizer = torch.optim.Adam(self.q_net.parameters(), lr=self.learning_rate)

        if self.use_prioritised_buffer:
            self.replay = PrioritisedReplayBuffer(self.buffer_size, self.alpha, self.beta_start, self.beta_frames )
        else:
            self.replay = ReplayBuffer(self.buffer_size)

    def _select_action(self, obs):
        # Epsilon-greedy
        if random.random() < self.epsilon:
            return np.array([self.action_space.sample() for _ in range(self.env.num_envs)])
        
        # Find best step (as model thinks) 
        obs_t = torch.tensor(obs, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            q_values = self.q_net(obs_t)
        return q_values.argmax(dim=1).cpu().numpy()

    def _train_step_prioritised(self):
        if len(self.replay) < self.batch_size:
            return
        obs, actions, rewards, next_obs, dones, indices, weights = self.replay.sample(self.batch_size)
        obs = obs.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_obs = next_obs.to(self.device)
        dones = dones.to(self.device)
        weights = weights.to(self.device)

        # Current Q values
        #picks the Q-value for the action actually taken
        current_q = self.q_net(obs).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q values
        with torch.no_grad():
            next_actions = self.q_net(next_obs).argmax(dim=1, keepdim=True)
            max_next_q = self.target_q_net(next_obs).gather(1, next_actions).squeeze(1)

        target_q = rewards + self.gamma * max_next_q * (1 - dones)

        td_errors = (current_q - target_q).detach().cpu().numpy()
        self.replay.update_priorities(indices, td_errors)

        loss = (weights * nn.functional.mse_loss(current_q, target_q, reduction="none")).mean()

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 10)  # gradient clipping
        self.optimizer.step()

        return loss.item()
    
    def _train_step(self):
        if len(self.replay) < self.batch_size:
            return

        obs, actions, rewards, next_obs, dones = self.replay.sample(self.batch_size)
        obs = obs.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_obs = next_obs.to(self.device)
        dones = dones.to(self.device)

        # Current Q values
        current_q = self.q_net(obs).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q values
        with torch.no_grad():
            max_next_q = self.target_q_net(next_obs).max(dim=1)[0]
            target_q = rewards + self.gamma * max_next_q * (1 - dones)

        loss = nn.functional.mse_loss(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 10)  # gradient clipping
        self.optimizer.step()

        return loss.item()

    def learn(self, total_timesteps, callback=None, **kwargs):
        self._setup_learn(total_timesteps, callback)
        
        # Eval env
        eval_env = make_vec_env(
        "tcs-v2",
        n_envs=1,
        env_kwargs={
            "height": 8,
            "width": 8,
            "obs_type": "data"},  
    )
        best_reward = -np.inf
        patience_counter = 0

        obs = self._last_obs
        episode = 0
        losses = []
        with trange(total_timesteps, desc="Training", unit="step") as pbar:
            while self.num_timesteps < total_timesteps:

                actions = self._select_action(obs)
                new_obs, rewards, dones, infos = self.env.step(actions)

                # Run episodes
                for i in range(self.env.num_envs):
                    self.replay.push(obs[i], actions[i], rewards[i], new_obs[i], float(dones[i]))

                obs = new_obs
                self._last_obs = new_obs
                self.num_timesteps += self.env.num_envs
                pbar.update(self.env.num_envs)

                # train
                if self.num_timesteps >= self.train_start:
                    if self.use_prioritised_buffer:
                        loss = self._train_step_prioritised()
                    else:
                        loss = self._train_step()
                        
                    if loss:
                        losses.append(loss)

                # Sync target network
                if self.num_timesteps % self.target_update == 0:
                    self.target_q_net.load_state_dict(self.q_net.state_dict())

                # Epsilon decay on episode end
                if any(dones):
                    episode += 1
                    self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

                # Logging
                if self.num_timesteps % 20000 == 0:
                    avg_loss = np.mean(losses[-100:]) if losses else 0
                    mean_reward, std_reward = evaluate_learning(self, eval_env, n_episodes=5)

                    # Early stopping check
                    if mean_reward > best_reward:
                        best_reward = mean_reward
                        patience_counter = 0
                    else:
                        patience_counter += 1
                        if patience_counter >= self.early_stopping_patience:
                            print(f"\nEarly stopping at step {self.num_timesteps}. No improvement for {self.early_stopping_patience} evals")
                            break

                    pbar.set_postfix({
                        "Episode": episode,
                        "Epsilon": f"{self.epsilon:.3f}",
                        "Loss": f"{avg_loss:.4f}",
                        "MeanReward": f"{mean_reward:.2f}",
                        "Best": f"{best_reward:.2f}",
                        "Patience": f"{patience_counter}/{self.early_stopping_patience}"
                    })

                    wandb.log({
                        "avg_loss": avg_loss,
                        "epsilon": self.epsilon,
                        "episode": episode,
                        "timestep": self.num_timesteps,
                        "mean_reward": mean_reward,
                        "std_reward": std_reward,
                        "best_reward": best_reward,
                        "patience_counter": patience_counter
                    })

        return self   
    

def evaluate_learning(model, env, n_episodes=5):

    rewards = []
    for _ in range(n_episodes):
        obs = env.reset()
        total = 0
        done = [False]
        while not done[0]:
            obs_t = torch.tensor(obs, dtype=torch.float32).to(model.device)
            with torch.no_grad():
                action = model.q_net(obs_t).argmax(dim=1).cpu().numpy()
            obs, r, done, _ = env.step(action)
            total += r[0]
        rewards.append(total)
    return np.mean(rewards), np.std(rewards)

If you decide to work on wrapper then you need to combine em into 1 class 

In [ ]:
import gymnasium as gym
# class FixObsWrapper(gym.ObservationWrapper):
#     """Pads observation to match the declared observation_space shape (9,)."""

#     def __init__(self, env):
#         super().__init__(env)
#         # Keep the declared space as-is (9,)

#     def observation(self, obs):
#         obs = np.array(obs, dtype=np.uint8)
#         if obs.shape[0] < 9:
#             obs = np.pad(obs, (0, 9 - obs.shape[0])) # pad with zeros
#         return obs
    

#no penalty - snake learns to survive
#too large a penalty - snake learns to die quickly
class RewardShapingWrapper(gym.Wrapper):
    def __init__(self, env, step_penalty=-0.01, hunger_cap=100):
        super().__init__(env)
        self.step_penalty = step_penalty
        self.hunger_cap = hunger_cap
        self.steps_since_food = 0

    def reset(self, **kwargs):
        self.steps_since_food = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)

        if reward >= 1:  # food eaten (your env gives +1 per food)
            self.steps_since_food = 0
        else:
            self.steps_since_food += 1

        if self.steps_since_food >= self.hunger_cap:
            terminated = True
            info["message"] = "Starved"
            self.steps_since_food = 0

        reward += self.step_penalty
        return obs, reward, terminated, truncated, info


    
    
# class CombinedWrapper(gym.Wrapper):
#     def __init__(self, env, step_penalty=-0.01,):
#         env = RewardShapingWrapper(env, step_penalty)
#         env = FixObsWrapper(env)
#         super().__init__(env)

In [20]:
import wandb
import os

    # wandb_key = os.getenv("WANDB_KEY")

try:
    wandb.login(key="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")

    api = wandb.Api()
    viewer = api.viewer
    print(f"Logged in as: {viewer.username}")
except wandb.errors.AuthenticationError:
    print("Not logged in")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Logged in as: danylo-zemskyi


Below is a single run pipeline, but use sweeps which are defined below

**still read my comments**

In [32]:
import numpy as np
import torch
from stable_baselines3.common.env_util import make_vec_env

def test_run():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
    print(f"Device: {DEVICE}")
    run = wandb.init()

    # NEVER EVER render mp4 here. It makes the whole training 4 times slower and GPU stalls
    # Maybe if use stablebaseline renderer then will be better
    vec_env = make_vec_env(
        "tcs-v2",
         #add more envs untill cpu cannot hadle it
        #cpu is the main bottleneck, since it has to sample buffer
        n_envs=16,
        env_kwargs={
            "height": 8,
            "width": 8,
            "obs_type": "data",
        },
        wrapper_class=RewardShapingWrapper,
        wrapper_kwargs={
            "step_penalty": -0.01,
        }
    )


    model = QLearning(
        env=vec_env,
        learning_rate=0.0005,   
        gamma=0.95, #don't over-value distant rewards
        epsilon_start=1.0,
        epsilon_end=0.01,
        epsilon_decay=0.9995,       
        buffer_size=200_000,        
        batch_size=256,
        train_start=5_000, # wait for more diverse experiences before training?
        target_update=1000,       
        alpha=0.6,                 
        beta_start=0.4,            
        beta_frames=500_000,       
        early_stopping_patience=50,
        device=DEVICE
    )


    model.learn(total_timesteps=1_000_000)

    torch.save(model.q_net.state_dict(), "qlearning.pt")
    print("Saved to qlearning.pt")

test_run()

Device: cuda


avg_loss,▁▆██▆▇▅▆▆▇▆▆▇▆▆▆▆▆▅▆▅▄▄▄▄▅▅▄▅▅▅▄▄▅▅▃▄▄▅▄
best_reward,▁▁▃▃▃▄▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████████
episode,▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
epsilon,█▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
mean_reward,▁▁▃▁▄▅▇▅▅▄▄▄▄▅▄▄▅▅▃▇█▆▄▆▄▃▃▇▅▂▆▆▃▅▃▃▃▃▃▂
patience_counter,▁▁▁▁▁▁▁▁▂▂▃▃▄▅▁▂▂▃▃▃▁▁▁▂▂▁▁▂▂▃▄▄▅▅▅▆▆▇▇█
std_reward,▃▃▃▂█▆▂▂▂▁▃▁▃▃▂▅▄▁▁▂▁▃▄▅▁▂▃▁▃▁▅▂▂▅▁▄▅▁▅▃
timestep,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
avg_loss,0.03825
best_reward,17
episode,16563


Using cuda device


Training:   0%|          | 0/1000000 [00:00<?, ?step/s]

Saved to qlearning.pt


In [ ]:
import wandb
from stable_baselines3.common.env_util import make_vec_env

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
print(f"Device: {DEVICE}")

def evaluate(model, env, n_episodes=10):
    rewards = []
    for _ in range(n_episodes):
        obs = env.reset()
        total = 0
        done = [False]
        while not done[0]:
            action = model.q_net(
                torch.tensor(obs, dtype=torch.float32).to(model.device)
            ).argmax(dim=1).cpu().numpy()
            obs, r, done, _ = env.step(action)
            total += r[0]
        rewards.append(total)
    return np.mean(rewards)

best_reward = -np.inf

def train():
    global best_reward
    run = wandb.init()
    config = run.config

    vec_env = make_vec_env(
        "tcs-v2",
         #add more envs untill cpu cannot hadle it
        #cpu is the main bottleneck, since it has to sample buffer
        n_envs=16,
        env_kwargs={
            "height": 8,
            "width": 8,
            "obs_type": "data",
        },
        wrapper_class=RewardShapingWrapper,
        wrapper_kwargs={
            "step_penalty": config.step_penalty,
        }
    )

    model = QLearning(
        env = vec_env,
        learning_rate = config.learning_rate,
        batch_size  = 256,
        gamma = config.gamma,
        epsilon_end=0.005,
        epsilon_decay = config.epsilon_decay,
        buffer_size = config.buffer_size,
        alpha=config.alpha,
        beta_start=config.beta_start,
        beta_frames=config.beta_frames,
        verbose = 0,
        device = DEVICE
    )

    model.learn(total_timesteps=2_000_000)
    
    #To prevent reward hacking. Penalty were leading the snake towards food
    #If this doesnt fix the reward hacking then remove distance-from-food penalty
    #in RewardShapingWrapper, or make it optional
    eval_env = make_vec_env(
        "tcs-v2",
        n_envs=1,
        env_kwargs={
            "height": 8,
            "width": 8,
            "obs_type": "data"},   
    )

    mean_reward = evaluate(model, vec_env, n_episodes=50)
    wandb.log({"mean_reward": mean_reward})

    # save only if best across entire sweep
    # if mean_reward > best_reward:
    best_reward = mean_reward
    model_path  = "best_sweep_model.pt"
    torch.save({
            "q_net": model.q_net.state_dict(),
            "mean_reward": best_reward,
            "config": dict(config),
        }, model_path)
    wandb.save(model_path)

        # Mark this run as the best in the sweep
    run.summary["best_reward"] = best_reward
    print(f"Run performance: {best_reward:.2f} | Run: {run.id}")

    vec_env.close()
    run.finish()


In [ ]:
# Grid search set up with wandb

sweep_config = {
    "method": "bayes", #possible values random, grid, and bayes just as seach classes
    "metric": {
        "name": "mean_reward",   
        "goal": "maximize"
    },
    "parameters": {
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-4,
            "max": 1e-2
            },
        "gamma": {
            "values": [0.998, 0.9975, 0.997]
            },
        "epsilon_decay": {
           "distribution": "log_uniform_values",
            "min": 0.9993,
            "max": 0.9999
            },
        "buffer_size": {
            "values": [250_000, 400_000, 500_000]
            },
        "step_penalty": {
            "distribution": "log_uniform_values",
            "min": -0.003,
            "max": -0.01
            },

        "alpha": {
            "values": [0.4, 0.6, 0.8]
        },
        "beta_start": {
            "values": [0.3, 0.4, 0.5]
        },
        "beta_frames": {
            "values": [50_000, 100_000, 200_000, 500_000]
        }
    }
}

# sweep_id = wandb.sweep(sweep_config, project="Snake_DRL",)
wandb.agent("danylo-zemskyi-city-university-of-london/Snake_DRL/6a2qacrw", function=train, count=50)

checkpoint = torch.load("best_sweep_model.pt", weights_only=False)
print(f"Best reward: {checkpoint['mean_reward']:.2f}")
print(f"Best config: {checkpoint['config']}")

q_net = DuelingQNetwork(obs_dim=9, n_actions=5)
q_net.load_state_dict(checkpoint["q_net"])
q_net.eval()

In [9]:
import wandb
import numpy as np


api = wandb.Api()
sweep = api.sweep("danylo-zemskyi-city-university-of-london/Snake_DRL/6a2qacrw")
best_run = max(sweep.runs, key=lambda r: r.summary.get("best_reward", -np.inf))

print(f"Best run: {best_run.id}")
print(f"Reward: {best_run.summary['mean_reward']:.2f}")
print(f"Config: {best_run.config}")

best_run.file("best_sweep_model.pt").download(replace=True)  # saves to current directory

Best run: dabx7vrg
Reward: 4.24
Config: {'alpha': 0.6, 'gamma': 0.997, 'beta_start': 0.3, 'beta_frames': 100000, 'buffer_size': 400000, 'step_penalty': -0.008634980599291527, 'epsilon_decay': 0.9997279966983073, 'learning_rate': 0.0009054169251761948}


<_io.TextIOWrapper name='.\\best_sweep_model.pt' mode='r' encoding='utf-8'>

Eval pipeline to generate video and eval results

In [ ]:
api = wandb.Api()

run = api.run("danylo-zemskyi-city-university-of-london/Snake_DRL/dabx7vrg")

# Download and load
run.file("best_sweep_model.pt").download(replace=True)

print(f"Reward : {run.summary.get('mean_reward'):.2f}")
print(f"Config : {run.config}")

Reward : 2.77
Config : {'alpha': 0.4, 'gamma': 0.9975, 'beta_start': 0.3, 'beta_frames': 50000, 'buffer_size': 400000, 'step_penalty': -0.008215193516757174, 'epsilon_decay': 0.9997752529812036, 'learning_rate': 0.006170108979689756}


In [38]:
import torch 
import numpy as np
from stable_baselines3.common.env_util import make_vec_env

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
print(f"Device: {DEVICE}")

# Create vectorized environment
vec_env = make_vec_env(
    "tcs-v2",
    env_kwargs={
        "height": 8,
        "width": 8,
        "obs_type": "data",
        "render_mode": "human",
        "mp4": True,
        "obs_type": "data"
    },
    n_envs=1,  
)

# Get dimensions
obs_dim = vec_env.observation_space.shape[0]
n_actions = vec_env.action_space.n

# Load Q-network
q_net = DuelingQNetwork(obs_dim, n_actions).to(DEVICE)
q_net.load_state_dict(torch.load("qlearning.pt", weights_only=False))
q_net.eval()

print(q_net.__dict__)
# Run episodes
n_episodes = 100
results = []

for ep in range(n_episodes):
    obs = vec_env.reset()
    total_reward = 0
    done = [False]

    while not done[0]:
        obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            action = q_net(obs_t).argmax(dim=1).cpu().numpy()
        obs, rewards, done, infos = vec_env.step(action)
        total_reward += rewards[0]

    results.append(total_reward)
    print(f"Episode {ep+1}: reward={total_reward:.2f}")
    
print(f"\nMean reward: {np.mean(results):.2f} | Std: {np.std(results):.2f}")
vec_env.close()

Device: cuda
{'training': False, '_parameters': {}, '_buffers': {}, '_non_persistent_buffers_set': set(), '_backward_pre_hooks': OrderedDict(), '_backward_hooks': OrderedDict(), '_is_full_backward_hook': None, '_forward_hooks': OrderedDict(), '_forward_hooks_with_kwargs': OrderedDict(), '_forward_hooks_always_called': OrderedDict(), '_forward_pre_hooks': OrderedDict(), '_forward_pre_hooks_with_kwargs': OrderedDict(), '_state_dict_hooks': OrderedDict(), '_state_dict_pre_hooks': OrderedDict(), '_load_state_dict_pre_hooks': OrderedDict(), '_load_state_dict_post_hooks': OrderedDict(), '_modules': {'feature': Sequential(
  (0): Linear(in_features=8, out_features=64, bias=True)
  (1): ReLU()
), 'value': Sequential(
  (0): Linear(in_features=64, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=1, bias=True)
), 'advantage': Sequential(
  (0): Linear(in_features=64, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=5, bias=T